## Initialization

### Retrieve Parameters

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

### Input Parameters

**report_args:** it is a json which contains all the appropriate parameters to run the report

`
{   
    "portfolio_name": ""MONZA PORTFOLIO - RETAIL",
    "output_file_prefix": "Monza_Retail_Daily_Extraction_1_",
    "report_output": "/Volumes/qa-cdh-delivery/internalreports/transactions-exports/DataExtraction1/MonzaRetail/"
}
`

**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
athens_tz = timezone('Europe/Athens')

try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("'", '"')
except Exception as e:
    raise ValueError("Missing required input: report_args")

try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")

try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")
    run_date_str = run_date.replace('-', '')
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

run_date = run_date.strftime('%Y-%m-%d')

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

portfolioName = "".join(json_data["portfolio_name"])
output_file_prefix = "".join(json_data["output_file_prefix"])
report_output = "".join(json_data["report_output"])

print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")

print(f"now is: {now_datetime_str}")

In [0]:
#athens_tz = timezone('Europe/Athens')

#env = 'qa'
#run_date = '2025-04-09'

RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{run_date}'), CAST('{run_date}' AS DATE)) AS DATE)").collect()[0][0]


##{    "portfolio_name": "MONZA PORTFOLIO - RETAIL",    "output_file_prefix": "Monza_Retail_Daily_Extraction_1_",    "report_output": "/Volumes/qa-cdh-delivery/internalreports/transactions-exports/DataExtraction1/MonzaRetail/" }

#Overwrite temporarily the values for testing purposes - DEBUG - To be deleted
output_file_prefix = "TransactionsExport_1_"
#report_output = "/Volumes/qa-cdh-delivery/internalreports/transactions-exports"

## Report Generation

### Exposure fields

In [0]:
# exposure information for maximum business date less or equal than running date
dexposure_ranked = spark.sql(f"""
            with rankedaccounts as (
                select  ExposureKeyQCAlpha as Account, 
                        AgreementKey as Contract_No, 
                        CDI as CDI,  
                        PartyTaxRegistrationNumber as AFM, 
                        ExposureStatus as Account_Status, 
                        physicalnumber as Physical_Number, 
                        OutstandingTotalBalance as TLC, 
                        GBV, 
                        DaysPastDueWithMateriality, 
                        DaysPastDue, 
                        DueAmount, 
                        IsSecuredFlag, 
                        LPS, 
                        NextInstallmentAmount,
                        NextInstallmentDate, 
                        SlaveFlag, 
                        InterestRate,
                        ActiveExposureRestructureStartDate, 
                        ExposureRestructureType,
                        ExposureRestructureOrigination,
                        BusinessDate as Last_Open_Date,
                        {run_date!r} as RunDate,
                        PortfolioName,
                        ExposureKey, 
                        ExposureKeyProfits,
                        CaseCode as Exp_CaseCode, 
                        ROW_NUMBER() OVER (PARTITION BY ExposureKeyQCAlpha ORDER BY BusinessDate DESC) AS rn
                from `{env}-cdh`.derivative.dexposure
                where   BusinessDate <= {run_date!r}
                    and isactiveflag = true
                    and ExposureKeyQCAlpha is not null
                    and PortfolioName ilike '%{portfolioName}%'
                    /*(   PortfolioName ILIKE '%MONZA PORTFOLIO - RETAIL%'
                        or  PortfolioName ILIKE '%DOMUS PORTFOLIO - RETAIL%'
                        or  PortfolioName ILIKE '%RHODIUM PORTFOLIO - RETAIL%')*/
            )
            select  * 
            from rankedaccounts
            where rn = 1

""")

dexposure_ranked.createOrReplaceTempView("dexposure_ranked")

### Assignment fields

Assignment Type definition

In [0]:
# assignments information for run date 
assignments = spark.sql(f"""
            with rankedaccounts as (
                select
                    a.BusinessDate, 
                    a.SourceSystem as Source, 
                    a.PortfolioName as Portfolio, 
                    a.ExposureKey as Assignments_ExposureKey,
                    b.Account,
                    a.ExternalCompanyFullName,
                    a.Channel,
                    a.SourceSystem, 
                    case 
                        when SourceSystem = 'qca' and AssignmentType = 'Channel Assignment' and Channel = 'Outsourcing Collection' then 'Collection'
                        when SourceSystem = 'qca' and AssignmentType = 'Channel Assignment' and Channel = 'Legal Agencies' then 'Modification_1'
                        when SourceSystem = 'qca' and AssignmentType = 'Channel Assignment' and Channel IN ('IRT', 'FAs') then 'Modification_2'
                        when SourceSystem = 'arotron' then 'Legal Actions'
                        else 'Else' 
                    END as Calculated_Type, 
                    a.AssignmentType,
                    a.AgentFullName,
                    a.Team,
                    a.CaseCode as Assignments_CaseCode,
                    ROW_NUMBER() OVER (PARTITION BY a.CaseCode ORDER BY a.BusinessDate DESC) AS rn
                from `{env}-cdh`.primary.assignment a
                inner join dexposure_ranked b
                    on a.CaseCode = b.Exp_CaseCode
                where   a.BusinessDate <= b.Last_Open_Date
                    and a.PortfolioName ilike '%{portfolioName}%'
                    /*(     a.PortfolioName ILIKE '%MONZA PORTFOLIO - RETAIL%'
                        or  a.PortfolioName ILIKE '%DOMUS PORTFOLIO - RETAIL%'
                        or  a.PortfolioName ILIKE '%RHODIUM PORTFOLIO - RETAIL%')*/
                )
                select  * 
                from rankedaccounts
                where rn = 1
""")

assignments.createOrReplaceTempView("assignments")

In [0]:
assignments_calc = spark.sql(f"""
                select  * , 
                    case
                        when Calculated_Type IN ('Collection', 'Modification_1') then ExternalCompanyFullName 
                        when Calculated_Type IN ('Modification_2', 'Legal Actions') then Channel 
                        else 'Else'
                    end as calc_Channel,
                    case
                        when Calculated_Type ='Collection' then 'N/A'
                        when Calculated_Type IN ('Modification_1', 'Modification_2') and Channel = 'FAs' then AgentFullName
                        else ''
                    end as calc_Agent,
                    case 
                        when Calculated_Type IN ('Modification_1', 'Modification_2') and Channel = 'FAs' then Team
                        else '' 
                    end as calc_Team
                from assignments
""")

assignments_calc.createOrReplaceTempView("assignments_calc")

In [0]:
assignments_max = spark.sql(f"""
                select  a.Assignments_ExposureKey,
                        a.Assignments_CaseCode,
                        max(a.BusinessDate) as LastAssignment_Date
                from assignments a
                group by a.Assignments_ExposureKey, a.Assignments_CaseCode
""")

assignments_max.createOrReplaceTempView("assignments_max")

### Interaction fields

In [0]:
# interaction information for run date 
interactions = spark.sql(f"""
                            select
                                i.ExposureKey as Interaction_ExposureKey,
                                MAX(i.CreatedDate) as Last_RPC_Date
                            from `{env}-cdh`.primary.interaction i
                            inner join dexposure_ranked b
                                on      i.ExposureKey = b.Account
                                    and i.BusinessDate  <= b.Last_Open_Date
                                    and i.CreatedDate   <= b.Last_Open_Date
                            where i.IsRPCFlag = true
                            group by i.ExposureKey
""")

interactions.createOrReplaceTempView("interactions")

### Payment fields

In [0]:
# payments information for run date 
payments = spark.sql(f"""
                        select
                            p.BusinessDate as Payments_Date, 
                            p.ExposureKey as Payments_ExposureKey, 
                            p.PortfolioName as Payments_PortfolioName, 
                            p.PaymentAmount, 
                            p.PaymentActualDate, 
                            p.PaymentTypeGroup,
                            b.Account 
                        from `{env}-cdh`.primary.payment p
                        inner join dexposure_ranked b
                            on      (p.ExposureKey = b.Account
                                or  p.ExposureKey = b.ExposureKeyProfits)
                        where   p.BusinessDate = {run_date!r}

""")

payments.createOrReplaceTempView("payments")

In [0]:
payments_sum = spark.sql(f"""
                        select
                            p.Payments_ExposureKey, 
                            sum(p.PaymentAmount) as TotalPayments
                        from payments p
                        where   p.PaymentActualDate <= {run_date!r}
                        group by p.Payments_ExposureKey
""")

payments_sum.createOrReplaceTempView("payments_sum")

In [0]:
last_cash_payment = spark.sql(f"""
                        select
                            p.Payments_ExposureKey, 
                            max(p.PaymentActualDate) as LastCashPaymentDate
                        from payments p
                        where   p.PaymentActualDate <= {run_date!r}
                            and p.PaymentTypeGroup = 'Cash'
                        group by p.Payments_ExposureKey
""")

last_cash_payment.createOrReplaceTempView("last_cash_payment")

In [0]:
payments_monthly = spark.sql(f"""
                        select
                            p.Payments_ExposureKey, 
                            sum(case when PaymentTypeGroup = 'Cash' then PaymentAmount else 0 end) as MTD_Cash_Payment, 
                            sum(PaymentAmount) as MTD_Total_Recoveries
                        from payments p
                        where   p.PaymentActualDate >= DATE_TRUNC('month', {run_date!r})
                            and p.PaymentActualDate <= {run_date!r}
                        group by p.Payments_ExposureKey
""")

payments_monthly.createOrReplaceTempView("payments_monthly")

In [0]:
last_assignments_payments_monthly = spark.sql(f"""
                        select
                            p.ExposureKey as Payments_ExposureKey, 
                            sum(p.PaymentAmount) as LastAssignment_MTD_CashPayments
                        from `{env}-cdh`.primary.payment p 
                        left join assignments_max am
                            on p.ExposureKey = am.Assignments_ExposureKey 
                        where   p.PaymentActualDate >= DATE_TRUNC('month', {run_date!r})
                            and p.PaymentActualDate >= am.LastAssignment_Date
                            and p.PaymentTypeGroup = 'Cash'
                        group by p.ExposureKey
""")

last_assignments_payments_monthly.createOrReplaceTempView("last_assignments_payments_monthly")

In [0]:
payments_quarterly = spark.sql(f"""
                        select
                            p.Payments_ExposureKey, 
                            sum(case when PaymentTypeGroup = 'Cash' then PaymentAmount else 0 end) as QTD_Cash_Payment, 
                            sum(PaymentAmount) as QTD_Total_Recoveries
                        from payments p
                        where   p.PaymentActualDate >= DATE_TRUNC('quarter', {run_date!r})
                            and p.PaymentActualDate <= {run_date!r}
                        group by p.Payments_ExposureKey
""")

payments_quarterly.createOrReplaceTempView("payments_quarterly")

In [0]:
last_assignments_payments_quarterly = spark.sql(f"""
                        select
                            p.ExposureKey as Payments_ExposureKey, 
                            sum(p.PaymentAmount) as LastAssignment_QTD_CashPayments
                        from `{env}-cdh`.primary.payment p
                        left join assignments_max am
                            on p.ExposureKey = am.Assignments_ExposureKey 
                        where  p.PaymentTypeGroup = 'Cash'
                                and p.PaymentActualDate >= DATE_TRUNC('quarter', {run_date!r})
                                and p.PaymentActualDate >= am.LastAssignment_Date                       
                        group by p.ExposureKey
""")

last_assignments_payments_quarterly.createOrReplaceTempView("last_assignments_payments_quarterly")

In [0]:
payment_fields = spark.sql(f"""
                        select distinct
                            ps.Payments_ExposureKey, 
                            lcp.LastCashPaymentDate,
                            coalesce(pm.MTD_Cash_Payment, 0) as MTD_Cash_Payment, 
                            coalesce(pm.MTD_Total_Recoveries, 0) as MTD_Total_Recoveries,
                            coalesce(pq.QTD_Cash_Payment,0) as QTD_Cash_Payment,
                            coalesce(pq.QTD_Total_Recoveries,0) as QTD_Total_Recoveries,
                            coalesce(lapm.LastAssignment_MTD_CashPayments,0) as LastAssignment_MTD_CashPayments,
                            coalesce(lapq.LastAssignment_QTD_CashPayments,0) as LastAssignment_QTD_CashPayments
                        from payments_sum ps
                        left outer join last_cash_payment lcp 
                            on ps.Payments_ExposureKey = lcp.Payments_ExposureKey
                        left outer join payments_monthly pm 
                            on ps.Payments_ExposureKey = pm.Payments_ExposureKey
                        left outer join payments_quarterly pq
                            on ps.Payments_ExposureKey = pq.Payments_ExposureKey
                        left outer join last_assignments_payments_monthly lapm
                            on ps.Payments_ExposureKey = lapm.Payments_ExposureKey 
                        left outer join last_assignments_payments_quarterly lapq
                            on ps.Payments_ExposureKey = lapm.Payments_ExposureKey 
""")

payment_fields.createOrReplaceTempView("payment_fields")


## Report Export

### Final export

In [0]:
final_export = spark.sql(f"""
                            select 
                              '2025-04-09' as Date, -- {run_date!r}
                              a.SourceSystem as Source,
                              r.PortfolioName as Portfolio,
                              r.Account,
                              r.Last_Open_Date,
                              r.Contract_No, 
                              r.CDI,  
                              r.AFM, 
                              r.Account_Status, 
                              r.Physical_Number, 
                              case when a.Calculated_Type like '%Modification%' then 'Modification' else a.Calculated_Type end as Assignment_Type,
                              a.calc_Channel as Channel,
                              a.calc_Team as Team, 
                              a.calc_Agent as Agent_Fa_Rm, 
                              r.TLC, 
                              r.GBV, 
                              r.DaysPastDueWithMateriality, 
                              r.DaysPastDue, 
                              r.DueAmount, 
                              r.IsSecuredFlag as SecuredFlag, 
                              r.LPS, 
                              r.NextInstallmentAmount,
                              r.NextInstallmentDate, 
                              r.SlaveFlag, 
                              r.InterestRate,
                              r.ActiveExposureRestructureStartDate, 
                              r.ExposureRestructureType,
                              r.ExposureRestructureOrigination,
                              p.MTD_Cash_Payment,
                              p.QTD_Cash_Payment,
                              p.LastCashPaymentDate,
                              p.MTD_Total_Recoveries,
                              p.QTD_Total_Recoveries,
                              P.LastAssignment_MTD_CashPayments,
                              P.LastAssignment_QTD_CashPayments,
                              i.Last_RPC_Date,
                              am.LastAssignment_Date
                            from dexposure_ranked r
                            left join assignments_calc a
                              on r.Exp_CaseCode = a.Assignments_CaseCode
                            left join interactions i 
                              on r.Account = i.Interaction_ExposureKey 
                            left join payment_fields p 
                              on r.Account = p.Payments_ExposureKey or r.ExposureKeyProfits = p.Payments_ExposureKey
                            left join assignments_max am
                              on r.Exp_CaseCode = am.Assignments_CaseCode
""")

final_export.createOrReplaceTempView("final_export")

In [0]:
final_report_df = final_export.toPandas()

#final_report_df['NextInstallmentAmount'] = final_report_df['NextInstallmentAmount'].apply(lambda x: round(x, 2))
#final_report_df['InterestRate'] = final_report_df['InterestRate'].apply(lambda x: f"{x:.2%}")
#final_report_df['MTD_Cash_Payment'] = final_report_df['MTD_Cash_Payment'].apply(lambda x: round(x, 2))

In [0]:
# Define the temporary file and output path
temp_file = f"/tmp/{output_file_prefix}{now_datetime_str}.xlsx"

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    final_report_df.to_excel(writer, sheet_name=f"{RefMonth.month}.{RefMonth.year}", index=False)

with open(temp_file, "wb") as f:
    f.write(output_buffer.getvalue())

output_path = f"{report_output}{run_date_str}/{output_file_prefix}{run_date}.xlsx"

# Ensure the directory exists
dbutils.fs.mkdirs(f"{report_output}{run_date_str}")

# Delete the output path if it exists
if os.path.exists(output_path):
    os.remove(output_path)

# Copy the temporary file to the final output path
shutil.copy(temp_file, output_path)

print(f"Output file is written successfully: {output_path}")

## Debug

In [0]:
%sql
--select count(Payments_ExposureKey), count(distinct Payments_ExposureKey) from payment_fields
--select count(Interaction_ExposureKey), count(distinct Interaction_ExposureKey) from interactions 
--select count(Assignments_ExposureKey), count(distinct Assignments_ExposureKey) from assignments_max 
--select count(ExposureKey), count(distinct ExposureKey) from dexposure_ranked 
--select count(distinct ExposureKey) from dexposure_ranked where account is not null 
-- 65669 -- 65669

In [0]:
%sql
--select * from final_export limit 10